In [30]:
%run "../../scripts/01-check_setup.ipynb"

SAP HANA Client for Python: 2.29.26061601
Using the dot-env file /Users/I076835/Repositories/GitHub/hana-ai-ve-kg-codejam/.env
Connected to SAP HANA db version 4.00.000.00.1782206579 (fa/CE2026.14) 
at c5889dd5-e0f6-4930-8408-94d53ca61dbf.hna0.prod-us10.hanacloud.ondemand.com:443 as CODEJAMHANAAI00
Current time on the SAP HANA server: 2026-07-11 12:36:04.522000


In [31]:
import glob
import json
import pandas as pd

html_files = sorted(glob.glob("agenda_*.html"))
print(f"Found {len(html_files)} file(s): {html_files}")

Found 3 file(s): ['agenda_hanatechcon_2026.html', 'agenda_recap_2026.html', 'agenda_ui5con_2026.html']


In [ ]:
rows = []
for file_path in html_files:
    with open(file_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    rows.append({"metadata": json.dumps({"file_name": file_path}), "content_html": html_content})
    print(f"  Loaded {len(html_content):,} chars from {file_path}")

df_event = pd.DataFrame(rows).convert_dtypes()
display(df_event.dtypes)
df_event

In [ ]:
from IPython.display import HTML

# Convert Markdown to HTML
html_output = df_event['content_html'].iloc[0]

display(HTML(html_output))

In [ ]:
import importlib
import config_events
importlib.reload(config_events)
from config_events import HANA_TABLE_NAME, HANA_SCHEMA_NAME

In [ ]:
from hana_ml.dataframe import create_dataframe_from_pandas

hdf_event_bronze = create_dataframe_from_pandas(
    connection_context=myconn,
    pandas_df=df_event,
    table_name=HANA_TABLE_NAME,
    schema=HANA_SCHEMA_NAME,
    force=True,
    object_type_as_bin=True,
    table_structure={"metadata": "NVARCHAR(5000)", "content_html": "NCLOB"}
)

In [ ]:
pd.set_option("max_colwidth", 256)
hdf_event_bronze.head(3).collect().T